# Notebook 07 — Export Pipeline (Data Contract)

**Project:** Planetary Defence Officer — ESA NEOCC Simulator  
**Purpose:** Produce the final `game_objects.json` consumed by the frontend. Implements the data contract from AGENT.md Section 4 exactly.

**Pool composition:**
- 250 real PHA
- 633 synthetic borderline hazardous
- 2000 real safe (nearest-neighbour sampled near hazardous objects)
- **Total: 2,883 objects**

**Sampling constraint:** Safe objects are selected by proximity to hazardous objects in feature space (10 nearest non-hazardous neighbours per hazardous), producing the numerically-similar-but-differently-classed clusters needed for the error mechanic.

---

## 1. Load all inputs

Real dataset (with Neo Reference IDs), synthetic pool, RF model, scaler.

In [ ]:
import numpy as np
import pandas as pd
import json, pickle, os, shutil, math
from datetime import datetime, timedelta
from sklearn.neighbors import NearestNeighbors

os.makedirs('output', exist_ok=True)
os.makedirs('../frontend/assets/data', exist_ok=True)

# ── Load artifacts ────────────────────────────────────────────────────────────
data       = np.load('output/dataset_full.npz', allow_pickle=True)
X_train_sc = data['X_train']
X_test_sc  = data['X_test']
y_train    = data['y_train']
y_test     = data['y_test']
ids_train  = data['ids_train']
ids_test   = data['ids_test']

with open('output/feature_names.json') as f:
    feature_names = json.load(f)

with open('output/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('output/rf_full.pkl', 'rb') as f:
    rf = pickle.load(f)

with open('output/synthetic_pool.json') as f:
    synthetic_pool = json.load(f)

# Raw CSV for close_approach dates (was dropped during preprocessing)
df_raw = pd.read_csv('data/raw/nasa.csv')
neo_id_to_date = (
    df_raw.drop_duplicates('Neo Reference ID')
          .set_index('Neo Reference ID')['Close Approach Date']
          .to_dict()
)

# ── Reconstruct real objects (inverse-transform scaled arrays) ────────────────
X_all_sc = np.vstack([X_train_sc, X_test_sc])
y_all    = np.concatenate([y_train, y_test])
ids_all  = np.concatenate([ids_train, ids_test]).astype(int)
X_all    = scaler.inverse_transform(X_all_sc)

df_real = pd.DataFrame(X_all, columns=feature_names)
df_real['label']          = y_all
df_real['neo_id']         = ids_all
df_real['close_approach'] = df_real['neo_id'].map(neo_id_to_date).fillna('2030-01-01')

print(f"Real objects: {len(df_real):,}  |  hazardous: {y_all.sum():,}  |  safe: {(y_all==0).sum():,}")
print(f"Synthetic pool: {len(synthetic_pool):,}")
print(f"Features ({len(feature_names)}): {feature_names}")

## 2. Run RF predictions on all real objects

In [ ]:
prob_all = rf.predict_proba(X_all_sc)[:, 1]
pred_all = rf.predict(X_all_sc)
df_real['model_confidence'] = prob_all
df_real['model_prediction'] = ['hazardous' if p == 1 else 'safe' for p in pred_all]
df_real['true_class']       = ['hazardous' if y == 1 else 'safe' for y in y_all]

acc = (pred_all == y_all).mean()
print(f"RF accuracy on full real dataset: {acc:.4f}")
print(f"Hazardous predicted: {pred_all.sum():,}  |  Hazardous true: {y_all.sum():,}")

## 3. Pool selection — 250 PHA + 2000 NN-safe

Nearest-neighbour sampling for the 2000 safe objects: each selected PHA contributes its 10 nearest safe neighbours, producing numerically-similar-but-differently-classed clusters.

In [ ]:
rng = np.random.default_rng(42)

df_pha_all  = df_real[df_real['label'] == 1].reset_index(drop=True)
df_safe_all = df_real[df_real['label'] == 0].reset_index(drop=True)

# 250 real PHAs (random sample, seed 42)
n_pha   = min(250, len(df_pha_all))
pha_idx = rng.choice(len(df_pha_all), size=n_pha, replace=False)
df_pha_pool = df_pha_all.iloc[pha_idx].copy()

# Map to global X_all_sc positions for NN search
pha_global_idx  = np.where(y_all == 1)[0]
safe_global_idx = np.where(y_all == 0)[0]
selected_pha_sc = X_all_sc[pha_global_idx[pha_idx]]
safe_all_sc_nn  = X_all_sc[safe_global_idx]

# For each selected PHA: 10 nearest safe neighbours in scaled feature space
nn = NearestNeighbors(n_neighbors=10, n_jobs=-1)
nn.fit(safe_all_sc_nn)
_, nbrs = nn.kneighbors(selected_pha_sc)   # (250, 10)

seen, ordered = set(), []
for idx in nbrs.flatten():
    if idx not in seen:
        seen.add(idx)
        ordered.append(idx)

global_safe = safe_global_idx[ordered]
if len(global_safe) < 2000:
    remaining   = np.setdiff1d(safe_global_idx, global_safe)
    extra       = rng.choice(remaining, size=2000 - len(global_safe), replace=False)
    global_safe = np.concatenate([global_safe, extra])
else:
    global_safe = global_safe[:2000]

df_safe_pool = df_real.iloc[global_safe].copy()

n_real_pha  = len(df_pha_pool)
n_real_safe = len(df_safe_pool)
n_syn       = len(synthetic_pool)

print(f"Real PHA:         {n_real_pha:,}")
print(f"Real safe:        {n_real_safe:,}")
print(f"Synthetic haz:    {n_syn:,}")
print(f"Total:            {n_real_pha + n_real_safe + n_syn:,}")

## 4. Assemble JSON per data contract (Section 4)

- `object_id` — real IDs for real objects; `YYYY-XX#` format for synthetic (years 2025–2032)
- `is_synthetic`, `true_class`, `model_prediction`, `model_confidence`
- `fields` nested by doc-type: orbital / proximity / physical / probability
- `source_attribution` — pseudo-random per object, seed=42
- `probability` fields (palermo, torino, impact_prob) are not in the raw dataset; generated as plausible synthetic values, seeded 42

In [ ]:
# ── Probability fields — plausible synthetic values (not in raw dataset) ──────
# Palermo scale: background objects ~ -12 to -6; monitored PHAs ~ -4 to -0.5
# Torino scale:  0 = routine; 1 = meriting attention; 2 = threatening
rng_prob = np.random.default_rng(42)

def gen_prob_fields(n, hazardous):
    if hazardous:
        palermo     = rng_prob.uniform(-4.0, -0.5, n).round(2).tolist()
        torino      = rng_prob.integers(1, 3, n).tolist()   # 1 or 2
        impact_prob = rng_prob.uniform(1e-5, 1e-2, n).round(8).tolist()
    else:
        palermo     = rng_prob.uniform(-12.0, -6.0, n).round(2).tolist()
        torino      = [0] * n
        impact_prob = rng_prob.uniform(1e-10, 1e-7, n).round(14).tolist()
    return palermo, torino, impact_prob

pha_palermo,  pha_torino,  pha_impact  = gen_prob_fields(n_real_pha,  True)
safe_palermo, safe_torino, safe_impact = gen_prob_fields(n_real_safe, False)
syn_palermo,  syn_torino,  syn_impact  = gen_prob_fields(n_syn,       True)

# ── Source attribution — pseudo-random per object, seed 42 ────────────────────
SOURCES = ['PAN-STARRS', 'ATLAS', 'CATALINA', 'JPL-CNEOS']
GROUPS  = ['orbital', 'proximity', 'physical', 'probability']
rng_src = np.random.default_rng(42)

def assign_sources(n):
    picks = rng_src.integers(0, 4, size=(n, 4))
    return [{g: SOURCES[p] for g, p in zip(GROUPS, row)} for row in picks]

pha_sources  = assign_sources(n_real_pha)
safe_sources = assign_sources(n_real_safe)
syn_sources  = assign_sources(n_syn)

# ── Synthetic object_ids — MPC provisional designation format ─────────────────
LETTERS = 'ABCDEFGHJKLMNOPQRSTUVWXYZ'   # no I (MPC convention)
rng_id  = np.random.default_rng(42)
rng_ca  = np.random.default_rng(42)

years = rng_id.integers(2025, 2033, n_syn)
l1s   = rng_id.integers(0, len(LETTERS), n_syn)
l2s   = rng_id.integers(0, len(LETTERS), n_syn)
nums  = rng_id.integers(1, 20, n_syn)
syn_ids = [f"{y}-{LETTERS[l1]}{LETTERS[l2]}{n}"
           for y, l1, l2, n in zip(years, l1s, l2s, nums)]

# ── Close-approach dates for synthetics — plausible future dates ──────────────
ca_base    = datetime(2025, 1, 1)
ca_offsets = rng_ca.integers(0, (datetime(2040, 12, 31) - ca_base).days, n_syn)
syn_dates  = [(ca_base + timedelta(days=int(d))).strftime('%Y-%m-%d') for d in ca_offsets]

print(f"Probability fields — PHA palermo: {min(pha_palermo):.2f}–{max(pha_palermo):.2f}")
print(f"Source attribution assigned for {n_real_pha + n_real_safe + n_syn:,} objects")
print(f"Synthetic IDs sample:   {syn_ids[:5]}")
print(f"Synthetic dates sample: {syn_dates[:5]}")

In [ ]:
def make_fields(a, e, i, q, Q, moid, close_approach, rel_v,
                H, dmin, dmax, palermo, torino, impact_prob):
    return {
        'orbital': {
            'a': round(float(a), 4), 'e': round(float(e), 4),
            'i': round(float(i), 4), 'q': round(float(q), 4),
            'Q': round(float(Q), 4),
        },
        'proximity': {
            'moid':           round(float(moid), 5),
            'close_approach': str(close_approach),
            'rel_v':          round(float(rel_v), 3),
        },
        'physical': {
            'H':            round(float(H), 2),
            'diameter_min': round(float(dmin), 5),
            'diameter_max': round(float(dmax), 5),
        },
        'probability': {
            'palermo':     round(float(palermo), 2),
            'torino':      int(torino),
            'impact_prob': float(impact_prob),
        },
    }

objects = []

# Real PHAs
for i, (_, row) in enumerate(df_pha_pool.iterrows()):
    objects.append({
        'object_id':          str(int(row['neo_id'])),
        'is_synthetic':       False,
        'true_class':         'hazardous',
        'model_prediction':   row['model_prediction'],
        'model_confidence':   round(float(row['model_confidence']), 4),
        'fields': make_fields(
            row['Semi Major Axis'], row['Eccentricity'], row['Inclination'],
            row['Perihelion Distance'], row['Aphelion Dist'],
            row['Minimum Orbit Intersection'], row['close_approach'],
            row['Relative Velocity km per sec'],
            row['Absolute Magnitude'], row['Est Dia in KM(min)'], row['Est Dia in KM(max)'],
            pha_palermo[i], pha_torino[i], pha_impact[i],
        ),
        'source_attribution': pha_sources[i],
    })

# Real safe
for i, (_, row) in enumerate(df_safe_pool.iterrows()):
    objects.append({
        'object_id':          str(int(row['neo_id'])),
        'is_synthetic':       False,
        'true_class':         'safe',
        'model_prediction':   row['model_prediction'],
        'model_confidence':   round(float(row['model_confidence']), 4),
        'fields': make_fields(
            row['Semi Major Axis'], row['Eccentricity'], row['Inclination'],
            row['Perihelion Distance'], row['Aphelion Dist'],
            row['Minimum Orbit Intersection'], row['close_approach'],
            row['Relative Velocity km per sec'],
            row['Absolute Magnitude'], row['Est Dia in KM(min)'], row['Est Dia in KM(max)'],
            safe_palermo[i], safe_torino[i], safe_impact[i],
        ),
        'source_attribution': safe_sources[i],
    })

# Synthetic hazardous
for i, s in enumerate(synthetic_pool):
    objects.append({
        'object_id':          syn_ids[i],
        'is_synthetic':       True,
        'true_class':         'hazardous',
        'model_prediction':   s['model_prediction'],
        'model_confidence':   round(float(s['model_confidence']), 4),
        'fields': make_fields(
            s['a'], s['e'], s['i'], s['q'], s['Q'],
            s['moid'], syn_dates[i], s['rel_v'],
            s['H'], s['diameter_min'], s['diameter_max'],
            syn_palermo[i], syn_torino[i], syn_impact[i],
        ),
        'source_attribution': syn_sources[i],
    })

print(f"Objects assembled: {len(objects):,}")
print(f"  Real hazardous:  {sum(1 for o in objects if not o['is_synthetic'] and o['true_class']=='hazardous')}")
print(f"  Real safe:       {sum(1 for o in objects if not o['is_synthetic'] and o['true_class']=='safe')}")
print(f"  Synthetic haz:   {sum(1 for o in objects if o['is_synthetic'])}")

## 5. JSON validation — no NaN, no Infinity, all required keys present

In [ ]:
REQUIRED_TOP    = {'object_id', 'is_synthetic', 'true_class',
                   'model_prediction', 'model_confidence',
                   'fields', 'source_attribution'}
REQUIRED_GROUPS = {'orbital', 'proximity', 'physical', 'probability'}
REQUIRED_FIELDS = {
    'orbital':     {'a', 'e', 'i', 'q', 'Q'},
    'proximity':   {'moid', 'close_approach', 'rel_v'},
    'physical':    {'H', 'diameter_min', 'diameter_max'},
    'probability': {'palermo', 'torino', 'impact_prob'},
}

errors = []
for j, obj in enumerate(objects):
    miss = REQUIRED_TOP - set(obj)
    if miss:
        errors.append(f"obj {j}: missing top-level {miss}"); continue
    miss = REQUIRED_GROUPS - set(obj['fields'])
    if miss:
        errors.append(f"obj {j}: missing field groups {miss}"); continue
    for grp, req in REQUIRED_FIELDS.items():
        miss = req - set(obj['fields'][grp])
        if miss:
            errors.append(f"obj {j} [{grp}]: missing {miss}")
    for grp in REQUIRED_GROUPS:
        for k, v in obj['fields'][grp].items():
            if isinstance(v, float) and (math.isnan(v) or math.isinf(v)):
                errors.append(f"obj {j} fields.{grp}.{k} = {v}")

if errors:
    print(f"VALIDATION FAILED — {len(errors)} error(s):")
    for e in errors[:20]:
        print(f"  {e}")
else:
    print(f"Validation PASSED — {len(objects):,} objects, all keys present, no NaN/Inf.")

## 6. Write output/game_objects.json and copy to frontend/assets/data/

In [ ]:
game_data = {
    'meta': {
        'generated_at':           '2026-05-16',
        'n_real_safe':            sum(1 for o in objects if not o['is_synthetic'] and o['true_class']=='safe'),
        'n_real_hazardous':       sum(1 for o in objects if not o['is_synthetic'] and o['true_class']=='hazardous'),
        'n_synthetic_hazardous':  sum(1 for o in objects if o['is_synthetic']),
        'model':                  'RandomForest_borderlineSMOTE',
        'model_accuracy':         0.9979,
        'model_pr_auc_hazardous': 1.0,
        'seed':                   42,
    },
    'objects': objects,
}

out_path      = 'output/game_objects.json'
frontend_path = '../frontend/assets/data/game_objects.json'

with open(out_path, 'w') as f:
    json.dump(game_data, f, indent=2)

shutil.copy2(out_path, frontend_path)

size_kb = os.path.getsize(out_path) / 1024
print(f"Saved  {out_path}  ({size_kb:.1f} KB)")
print(f"Copied {frontend_path}")

## 7. Summary statistics

In [ ]:
pha_conf  = [o['model_confidence'] for o in objects if not o['is_synthetic'] and o['true_class']=='hazardous']
syn_conf  = [o['model_confidence'] for o in objects if o['is_synthetic']]
safe_conf = [o['model_confidence'] for o in objects if o['true_class']=='safe']

print(f"\n◈ EXPORT COMPLETE — game_objects.json")
print(f"{'='*55}")
print(f"  Total objects:           {len(objects):,}")
print(f"  Real safe:               {game_data['meta']['n_real_safe']:,}")
print(f"  Real hazardous:          {game_data['meta']['n_real_hazardous']:,}")
print(f"  Synthetic hazardous:     {game_data['meta']['n_synthetic_hazardous']:,}")
print(f"{'='*55}")
print(f"  Haz conf   mean={np.mean(pha_conf):.3f}  min={np.min(pha_conf):.3f}  max={np.max(pha_conf):.3f}")
print(f"  Syn conf   mean={np.mean(syn_conf):.3f}  min={np.min(syn_conf):.3f}  max={np.max(syn_conf):.3f}")
print(f"  Safe conf  mean={np.mean(safe_conf):.3f}  min={np.min(safe_conf):.3f}  max={np.max(safe_conf):.3f}")

# Round-trip verification
with open('output/game_objects.json') as f:
    verify = json.load(f)
assert len(verify['objects']) == len(objects), "Round-trip count mismatch!"
print(f"\nRound-trip PASSED — {len(verify['objects']):,} objects verified in file.")
print(f"Phase 6 complete.")